# Week 2 — Day 4: Compare fine-tuning experiments

Tasks:
- Run 2-3 fine-tuning/config experiments (learning rates, decoding strategies) — done on Kaggle,
  see `notebooks/kaggle_week2_finetune_blip.ipynb`
- Log each run to MLflow; compare metrics across runs

Deliverable: MLflow dashboard with 3+ compared runs (zero-shot baseline + 3 fine-tuning configs);
a short note on which configuration performed best and why.

**Prerequisite:** run `notebooks/kaggle_week2_finetune_blip.ipynb` on Kaggle first, then download
`week2_finetune_results.json` (and `.csv`) from its Output tab into this repo's `data/processed/`
folder. This notebook reads that file — it will raise a clear error if it's not there yet.

### 1. Load the Kaggle fine-tuning results

In [ ]:
import os
import json
import mlflow
import pandas as pd

results_path = '../data/processed/week2_finetune_results.json'
if not os.path.exists(results_path):
    raise FileNotFoundError(
        "week2_finetune_results.json not found in data/processed/.\n"
        "Run notebooks/kaggle_week2_finetune_blip.ipynb on Kaggle first, then download its "
        "output JSON/CSV into data/processed/ before running this notebook."
    )

with open(results_path) as f:
    finetune_results = json.load(f)

finetune_df = pd.DataFrame(finetune_results).T
finetune_df

### 2. Log each fine-tuning config as an MLflow run

In [ ]:
mlflow.set_tracking_uri("sqlite:///../mlflow.db")
mlflow.set_experiment("flickr8k-image-captioning")

logged_run_ids = {}
for config_name, row in finetune_df.iterrows():
    with mlflow.start_run(run_name=config_name) as run:
        mlflow.log_params({
            "model_name": "Salesforce/blip-image-captioning-base",
            "fine_tuned": True,
            "vision_encoder_frozen": True,
            "learning_rate": row["lr"],
            "decoding_strategy": row["decoding"],
            "trained_on": "kaggle-gpu",
        })
        mlflow.log_metrics({
            "bleu": row["bleu"],
            "rouge1": row["rouge1"],
            "rouge2": row["rouge2"],
            "rougeL": row["rougeL"],
            "final_train_loss": row["final_train_loss"],
            "train_time_s": row["train_time_s"],
        })
        logged_run_ids[config_name] = run.info.run_id

logged_run_ids

### 3. Compare all runs (zero-shot baseline + fine-tuned configs)

In [ ]:
client = mlflow.tracking.MlflowClient()
experiment = client.get_experiment_by_name("flickr8k-image-captioning")
all_runs = client.search_runs([experiment.experiment_id], order_by=["metrics.bleu DESC"])

comparison = pd.DataFrame([
    {
        "run_name": r.data.tags.get("mlflow.runName", r.info.run_id[:8]),
        "fine_tuned": r.data.params.get("fine_tuned"),
        "decoding": r.data.params.get("decoding_strategy"),
        "lr": r.data.params.get("learning_rate", "-"),
        "bleu": r.data.metrics.get("bleu"),
        "rouge1": r.data.metrics.get("rouge1"),
        "rougeL": r.data.metrics.get("rougeL"),
    }
    for r in all_runs
])
comparison

### 4. Pick the baseline going forward

Fill in after reviewing the comparison table above:

- **Best config:** *(name + BLEU/ROUGE)*
- **Why:** *(e.g., best BLEU without a large drop in ROUGE; or beam search improved fluency at
  acceptable extra latency; or the lower learning rate reduced final train loss without overfitting)*
- This becomes the model used for Week 3's XAI analysis and gets registered in the MLflow Model
  Registry in Week 4.

To view all runs side-by-side in the UI:
```
.venv\Scripts\mlflow ui --backend-store-uri sqlite:///mlflow.db
```
then open http://127.0.0.1:5000